# VeriLens Research Platform — DistilBERT training (Colab GPU from VS Code)

This notebook trains fresh DistilBERT checkpoints for the **random** and **chronological** research protocols, evaluates them on the matching held-out test sets, and fits optional temperature scaling on validation data only.

**Important:** the Colab runtime is remote. Your Windows `D:\FakeNewsResearchPlatform` folder is not automatically visible to it. First copy the whole project (including `data/raw/Fake.csv` and `data/raw/True.csv`) into `MyDrive/FakeNewsResearchPlatform`, then select the **Colab GPU** kernel for this notebook in VS Code.

Labels are fixed throughout: `0 = Fake News`, `1 = True News`. Keep the test set untouched. A classifier prediction is not independent fact verification.


## 1. Mount Drive and configure paths

In VS Code run `Ctrl+Shift+P` → **Colab: Mount Google Drive to Server...** and execute the inserted mount cell. Copy the complete project folder and both raw CSVs into `MyDrive/FakeNewsResearchPlatform` before proceeding. This works with the official VS Code Colab extension without depending on the local Windows D: filesystem being visible remotely.


In [ ]:
from pathlib import Path
import shutil, os, sys, subprocess, json, time

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/FakeNewsResearchPlatform')
WORK_DIR = Path('/content/FakeNewsResearchPlatform')

if not Path('/content/drive/MyDrive').exists():
    raise RuntimeError('Google Drive is not mounted. Use Ctrl+Shift+P → Colab: Mount Google Drive to Server..., execute the inserted cell, then rerun this cell.')
if not (DRIVE_PROJECT_DIR / 'backend' / 'training' / 'train_distilbert.py').exists():
    raise FileNotFoundError(f'Project files not found at {DRIVE_PROJECT_DIR}. Copy the project folder into MyDrive/FakeNewsResearchPlatform first.')
if not (DRIVE_PROJECT_DIR / 'data' / 'raw' / 'Fake.csv').exists() or not (DRIVE_PROJECT_DIR / 'data' / 'raw' / 'True.csv').exists():
    raise FileNotFoundError('Missing data/raw/Fake.csv and/or data/raw/True.csv in your Drive project folder.')

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(DRIVE_PROJECT_DIR, WORK_DIR, ignore=shutil.ignore_patterns('.venv', 'venv', 'node_modules', '.env', '*.sqlite3', '__pycache__', '.git'))
os.chdir(WORK_DIR)
print('Drive project:', DRIVE_PROJECT_DIR)
print('Working copy:', WORK_DIR)
print('Fake.csv exists:', (WORK_DIR / 'data/raw/Fake.csv').exists())
print('True.csv exists:', (WORK_DIR / 'data/raw/True.csv').exists())


## 2. Verify that this notebook is using a GPU

Do not start full training unless a CUDA GPU is visible. Runtime availability depends on your Colab account, quota and the selected kernel.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU visible. In VS Code select the Colab GPU kernel, reconnect, and run this cell again.')


## 3. Install the notebook training dependencies

PyTorch is supplied by the Colab runtime. The requirements file pins the other main experiment libraries used by the project.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK_DIR / 'backend' / 'requirements-colab.txt')], check=True)


## 4. Prepare the dataset and make research splits

The preparation step removes short/empty rows, removes exact normalized duplicates, excludes exact texts with conflicting labels, and creates an audit report. The random and temporal protocols are separate experiments; temporal label prevalence may differ substantially.


In [ ]:
def run_module(module, *args):
    command = [sys.executable, '-m', module, *map(str, args)]
    print('+', ' '.join(command))
    subprocess.run(command, cwd=WORK_DIR, check=True)

run_module('backend.training.prepare_dataset')
run_module('backend.training.create_research_splits')
run_module('backend.training.validate_splits')

for protocol in ('random', 'temporal'):
    base = WORK_DIR / 'data' / 'processed' / 'research_splits' / protocol
    print(protocol, {name: len(__import__('pandas').read_csv(base / f'{name}.csv')) for name in ('train', 'validation', 'test')})


## 5. Train and evaluate the TF-IDF baseline

Save these results first. They provide a simpler reference for comparison with DistilBERT.


In [ ]:
for protocol in ('random', 'temporal'):
    split = WORK_DIR / 'data' / 'processed' / 'research_splits' / protocol
    model = DRIVE_PROJECT_DIR / 'backend' / 'models' / f'tfidf_{protocol}.joblib'
    run_module('backend.training.train_tfidf_baseline', '--protocol', protocol, '--output-model', str(model))
    run_module('backend.training.evaluate_model', '--model-type', 'tfidf', '--model-path', str(model),
               '--test-csv', str(split / 'test.csv'), '--output-prefix',
               str(WORK_DIR / 'outputs' / 'research' / f'evaluation_tfidf_{protocol}'))


## 6. Optional GPU smoke test (recommended before full training)

This trains on only 128 training examples for one epoch. It checks that tokenization, GPU, training and model saving work; **do not report the smoke-test model as the main result**.


In [ ]:
SMOKE_MODEL = DRIVE_PROJECT_DIR / 'backend' / 'models' / 'smoke_distilbert'
run_module('backend.training.train_distilbert', '--protocol', 'random', '--epochs', '1',
           '--batch-size', '8', '--max-length', '128', '--max-train-rows', '128',
           '--output-dir', str(SMOKE_MODEL))
print('Smoke-test checkpoint:', SMOKE_MODEL)


## 7. Full DistilBERT training — random split

This is a full training run. It may take substantial GPU time. The model is written to Google Drive so the best checkpoint from each epoch can persist if the Colab session disconnects. For final work, record the actual runtime, package versions and chosen settings.


In [ ]:
RANDOM_MODEL = DRIVE_PROJECT_DIR / 'backend' / 'models' / 'research_distilbert_random'
run_module('backend.training.train_distilbert', '--protocol', 'random', '--epochs', '2',
           '--batch-size', '8', '--max-length', '256', '--seed', '42',
           '--output-dir', str(RANDOM_MODEL))


### Persist the random run report to Drive


In [ ]:
drive_outputs = DRIVE_PROJECT_DIR / 'outputs' / 'research'
drive_outputs.mkdir(parents=True, exist_ok=True)
local_outputs = WORK_DIR / 'outputs' / 'research'
if local_outputs.exists():
    shutil.copytree(local_outputs, drive_outputs, dirs_exist_ok=True)
print('Reports copied to', drive_outputs)


## 8. Full DistilBERT training — temporal split

The chronological experiment uses only dated rows ordered by publication date. It is important to report class counts and label-prevalence drift. The temporal test is not interchangeable with the random test.


In [ ]:
TEMPORAL_MODEL = DRIVE_PROJECT_DIR / 'backend' / 'models' / 'research_distilbert_temporal'
run_module('backend.training.train_distilbert', '--protocol', 'temporal', '--epochs', '2',
           '--batch-size', '8', '--max-length', '256', '--seed', '42',
           '--output-dir', str(TEMPORAL_MODEL))


## 9. Evaluate on the matching locked test sets

This evaluates random-trained DistilBERT on the random test and temporal-trained DistilBERT on the temporal test. Do not tune parameters after examining final test scores.


In [ ]:
for protocol, model_path in (('random', RANDOM_MODEL), ('temporal', TEMPORAL_MODEL)):
    test_csv = WORK_DIR / 'data' / 'processed' / 'research_splits' / protocol / 'test.csv'
    run_module('backend.training.evaluate_model', '--model-type', 'transformer', '--model-path', str(model_path),
               '--test-csv', str(test_csv), '--max-length', '256', '--batch-size', '16',
               '--output-prefix', str(WORK_DIR / 'outputs' / 'research' / f'evaluation_distilbert_{protocol}'))


## 10. Optional temperature scaling (fit on validation only)

Temperature scaling can improve probability calibration, but it is not guaranteed to do so on every dataset. Fit one temperature per model using its validation split, then evaluate the frozen temperature once on its matching test set.


In [ ]:
for protocol, model_path in (('random', RANDOM_MODEL), ('temporal', TEMPORAL_MODEL)):
    validation_csv = WORK_DIR / 'data' / 'processed' / 'research_splits' / protocol / 'validation.csv'
    temp_json = WORK_DIR / 'outputs' / 'research' / f'temperature_transformer_{protocol}.json'
    run_module('backend.training.calibrate_temperature', '--model-type', 'transformer', '--model-path', str(model_path),
               '--validation-csv', str(validation_csv), '--max-length', '256', '--batch-size', '16',
               '--output-json', str(temp_json))
    test_csv = WORK_DIR / 'data' / 'processed' / 'research_splits' / protocol / 'test.csv'
    run_module('backend.training.evaluate_model', '--model-type', 'transformer', '--model-path', str(model_path),
               '--test-csv', str(test_csv), '--max-length', '256', '--batch-size', '16',
               '--temperature-json', str(temp_json),
               '--output-prefix', str(WORK_DIR / 'outputs' / 'research' / f'evaluation_distilbert_{protocol}_calibrated'))


## 11. Optional Captum Saliency faithfulness audit

This perturbs the highest-attribution tokens and compares the resulting model-logit change against random-token and lowest-attribution controls. It is an exploratory **model-sensitivity** test, not proof the explanation is correct or that the article is true. It can take several minutes on a GPU. Run after both DistilBERT checkpoints exist; skip if your runtime budget is limited.


In [ ]:
for protocol, model_path in (('random', RANDOM_MODEL), ('temporal', TEMPORAL_MODEL)):
    test_csv = WORK_DIR / 'data' / 'processed' / 'research_splits' / protocol / 'test.csv'
    out_json = WORK_DIR / 'outputs' / 'research' / f'explanation_faithfulness_{protocol}.json'
    run_module('backend.training.evaluate_explanation_faithfulness', '--model-path', str(model_path),
               '--test-csv', str(test_csv), '--samples', '50', '--max-length', '256',
               '--mask-fraction', '0.10', '--output-json', str(out_json))


## 12. Save reports to Google Drive

Model checkpoint folders were written directly to Drive. Copy metrics, predictions, split reports and training logs back to Drive, too.


In [ ]:
if local_outputs.exists():
    shutil.copytree(local_outputs, drive_outputs, dirs_exist_ok=True)
print('Models:')
print(RANDOM_MODEL)
print(TEMPORAL_MODEL)
print('Reports:', drive_outputs)


## 13. Verify predictions with Google Fact Check Tools API (local app)

The Google Fact Check Tools API is used by the FastAPI application, not as a training dataset. After the model checkpoints are copied to `D:\\FakeNewsResearchPlatform\\backend\\models`, configure `GOOGLE_FACTCHECK_API_KEY` in the project-root `.env` and restart FastAPI.

1. Open `http://127.0.0.1:8000/docs` and check `GET /api/health` reports the Google API as configured.
2. Try `GET /api/fact-check/search?q=...` with a concise public claim.
3. Submit a headline, article text and a concise Evidence search query in the React Article Analysis page. `POST /api/analyze` returns the model prediction and Google ClaimReview results separately, plus related GDELT news coverage.
4. Record query count, result/no-result/API-error status, publisher, textual rating, review date, and reviewer-assessed relevance for the research report.

**Interpretation:** Google returns indexed published ClaimReview entries only. A returned rating must be read in context, and a no-match response means `Unverified`, not true or false. Google results never automatically overwrite the classifier prediction.


## Research reporting checklist

- [ ] Save `outputs/research/dataset_preparation_report.json` and `data/processed/research_splits/split_report.json`.
- [ ] Report random and temporal class counts and date ranges.
- [ ] Compare TF-IDF and DistilBERT using macro-F1, MCC, ROC-AUC, ECE, Brier score, confusion matrix and per-class precision/recall.
- [ ] Fit any calibration parameters on validation data only.
- [ ] Exclude all validation/test overlaps when using expert-approved feedback.
- [ ] Record Google Fact Check API query coverage, response status, publisher/rating/date details and human relevance review; do not use no-match results as labels.
- [ ] Record model versions, random seeds, package versions and failed/negative results too.

A successful training run is not itself evidence of research novelty. Report measured results only after the scripts execute successfully on your chosen GPU runtime.
